# Reto Incendios

Dos pasos separados:

| Paso | Qué hace | Red |
|---|---|---|
| **1 · Crawling** | Descarga con crawl4ai los focos de satélite, las áreas quemadas, las incidencias 112 y el riesgo de AEMET | sí |
| **2 · Datos** | Revisa cada capa en tablas, la guarda en `retos/output/incendios/<fecha>/` y comprueba que los ficheros están completos | no |

| Capa | Fuente | Clave |
|---|---|---|
| Focos de calor | NASA FIRMS (VIIRS, MODIS) · EUMETSAT LSA-SAF (SEVIRI) | SEVIRI: usuario LSA-SAF |
| Áreas quemadas | Copernicus EFFIS | no |
| Incidencias 112 | INFOCA · INFOCAM/FIDIAS · Bombers · INFORCYL | no |
| Riesgo previsto | AEMET (mapa oficial georreferenciado, 3 días) | AEMET |
| Cobertura del suelo en cada foco | ESA WorldCover (datos locales) | no (generar una vez) |

Solo produce datos, no abre ningún mapa: para verlos en el mapa interactivo está el pipeline completo (`pipeline_incendios.ipynb`).

## 0. Preparación

In [ ]:
# Recarga automática: si el código de incendios/ o retos/ cambia, se usa la versión nueva sin reiniciar.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "incendios" / "__init__.py").exists()), None)
if RAIZ is None:
    raise FileNotFoundError("No encuentro la carpeta incendios/: abre el notebook desde Incendios-V0.0/ o retos/")
for ruta in (RAIZ, RAIZ / "retos"):
    if str(ruta) not in sys.path:
        sys.path.insert(0, str(ruta))

import pandas as pd

from incendios.config import fijar_claves, estado_claves
from incendios.pipeline import resumen
from reto_incendios import comprobar, crawlear, guardar, leer_json, nueva_carpeta

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print("Proyecto:", RAIZ)

## 1. Claves

- **AEMET:** necesaria para el riesgo previsto.
- **LSA-SAF:** necesaria para los focos de Meteosat SEVIRI.
- **Sin ellas:** FIRMS, EFFIS y las incidencias 112 funcionan igual.

Si ya están en `.env` (por ejemplo, porque las guardaste desde el pipeline), deja las comillas vacías. ⚠️ Si las pegas aquí, no subas este notebook a git.

In [ ]:
AEMET_API_KEY   = ""
LSASAF_USUARIO  = ""   # opcional (SEVIRI)
LSASAF_PASSWORD = ""   # opcional (SEVIRI)

fijar_claves(aemet=AEMET_API_KEY, lsasaf_usuario=LSASAF_USUARIO, lsasaf_password=LSASAF_PASSWORD)
for variable, valor in estado_claves().items():
    print(f"{variable:<16} {valor}")

## 2. Parámetros

In [ ]:
HORAS_FOCOS      = 24    # ventana de focos FIRMS (24, 48 o hasta 168 h)
HORAS_SEVIRI     = 3     # ventana de focos SEVIRI (4 ficheros por hora)
DIAS_EFFIS       = 7     # áreas quemadas publicadas o revisadas
DIAS_INCIDENCIAS = 3     # incidencias 112 ya extinguidas que se incluyen

# PASO 1 · Crawling

Las cinco fuentes se lanzan en paralelo con crawl4ai. Cada una va aislada: si falla o le falta la clave, queda anotada como `error` u `omitida` y las demás siguen.

### 1.1 Ejecutar el crawling

In [ ]:
parametros = {"horas_focos": HORAS_FOCOS, "horas_seviri": HORAS_SEVIRI,
              "dias_effis": DIAS_EFFIS, "dias_incidencias": DIAS_INCIDENCIAS}
resultado = await crawlear(**parametros)
print(resumen(resultado["estado"]))

# PASO 2 · Datos

Revisión de cada capa, guardado en ficheros (GeoJSON, imágenes de riesgo, `estado.json` y `manifiesto.json`) y comprobación de que se han escrito completos. No hace peticiones de red: trabaja con lo que ha descargado el paso 1.

### 2.1 Focos de calor

Color por confianza (alta/media/baja) y `frp_mw` = potencia radiativa del fuego. SEVIRI no ve a través de las nubes: el resumen de arriba indica qué porcentaje de España pudo vigilar.

In [ ]:
df_focos = pd.DataFrame(resultado["focos"])
if df_focos.empty:
    print("Sin focos en la ventana.")
else:
    display(df_focos.groupby(["sensor", "satelite", "confianza"]).size().rename("focos").to_frame())
    display(df_focos.sort_values("frp_mw", ascending=False)[["fecha_utc", "satelite", "lat", "lon", "confianza", "frp_mw"]].head(10))

### 2.2 Cobertura del suelo en cada foco

El paso 1 anota en cada foco y en cada incidencia qué hay en el suelo a su alrededor (ESA WorldCover, datos locales). Así se separa la vegetación forestal de los cultivos, las zonas urbanas o el agua. Si la fuente aparece `omitida` en el resumen, hay que generar los datos una vez con `python herramientas/generar_cobertura.py`.

In [ ]:
if "cobertura" in df_focos:
    con = df_focos[df_focos["cobertura"].notna()]
    tabla_cob = pd.DataFrame({"fecha_utc": con["fecha_utc"], "satelite": con["satelite"], "frp_mw": con["frp_mw"],
                              "suelo dominante": con["cobertura"].map(lambda c: c["dominante"]),
                              "vegetación natural %": con["cobertura"].map(lambda c: c["combustible_pct"])})
    display(tabla_cob.sort_values("frp_mw", ascending=False))
    display(tabla_cob["suelo dominante"].value_counts().rename("focos").to_frame())
else:
    print("Sin cobertura del suelo en esta ejecución (ver el resumen del paso 1).")

### 2.3 Áreas quemadas (EFFIS)

Cuenta lo que EFFIS ha **publicado o revisado** en la ventana. La columna `novedad` distingue:
- **`nueva`:** el incendio cae dentro de la ventana.
- **`actualizada`:** incendio anterior cuyo perímetro EFFIS ha revisado.

In [ ]:
df_areas = pd.DataFrame(resultado["areas"])
if df_areas.empty:
    print("EFFIS no ha publicado áreas en la ventana.")
else:
    print(df_areas["novedad"].value_counts().to_dict(), "·", df_areas["area_ha"].sum(), "ha")
    display(df_areas[["fecha_incendio", "novedad", "provincia", "municipio", "area_ha", "pct_natura2000"]]
            .sort_values("area_ha", ascending=False).head(10))

### 2.4 Incidencias 112

Las publican los servicios de emergencia autonómicos. Un "aviso sin fase" significa que el servicio ha abierto el aviso pero no lo ha clasificado; no confirma un incendio activo.

In [ ]:
print(resultado["estado"]["Incidencias 112 (oficiales)"].get("detalle_fuentes"))
df_inc = pd.DataFrame(resultado["incidencias"])
if not df_inc.empty:
    display(df_inc[["comunidad", "municipio", "provincia", "estado", "inicio_utc", "superficie_ha", "fuente"]])

### 2.5 Riesgo previsto (AEMET)

In [ ]:
for r in resultado["riesgo"]:
    print(f"{r['fecha']}: nivel máximo {r['nivel_max']} · reparto (%) {r['reparto_pct']}")
if not resultado["riesgo"]:
    print("Sin riesgo: falta la clave de AEMET o la fuente falló (ver resumen).")

### 2.6 Guardar los ficheros

Escribe en una carpeta nueva los GeoJSON, las imágenes de riesgo, `estado.json` y `manifiesto.json`. El manifiesto indica qué fichero es cada capa y cuántos registros tiene, para que cualquier otra aplicación pueda leerlos.

In [ ]:
carpeta = nueva_carpeta("incendios")
guardar(resultado, carpeta, parametros)
print("Guardado en", carpeta)
manifiesto = leer_json(carpeta / "manifiesto.json")
display(pd.DataFrame(manifiesto["capas"]).T[["fichero", "tipo", "registros"]])

### 2.7 Comprobar los ficheros guardados

Se relee lo que se acaba de escribir y se compara con el manifiesto y con lo que hay en memoria: cada fichero tiene que existir, poder leerse y tener todos sus registros.
- **Todo ✓:** los datos están completos.
- **Algún ✗:** vuelve a ejecutar la sección anterior.

In [ ]:
tabla = pd.DataFrame(comprobar(carpeta))
tabla["ok"] = tabla["ok"].map({True: "✓", False: "✗"})
display(tabla)
print("Datos completos ✓" if (tabla["ok"] == "✓").all() else "Hay ficheros incompletos ✗: vuelve a guardar")
print()
print("Carpeta de datos:", carpeta)
for f in sorted(carpeta.rglob("*")):
    if f.is_file():
        print(f"   {f.relative_to(carpeta)}  ({f.stat().st_size // 1024} KB)")

### Mapa interactivo

Genera `mapa_incendios.html` en la carpeta de datos y lo abre en el navegador. Lee solo los ficheros guardados: no hace peticiones.

In [ ]:
from reto_incendios import mapa
ruta = mapa(carpeta, abrir=True)
print("Mapa:", ruta)